<a href="https://colab.research.google.com/github/zune2/DSpython/blob/main/2026-10-04-ep001-vector-db-recommend-sys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install faiss-cpu sentence-transformers numpy

In [7]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [8]:
# 1. Hugging Face에서 무료 오픈소스 한국어 임베딩 모델 로드
# Kaggle의 GPU 메모리에 모델을 올립니다.
print("🔄 Hugging Face로부터 오픈소스 임베딩 모델을 로드하는 중...")
model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print("✅ 모델 로드 완료!")

🔄 Hugging Face로부터 오픈소스 임베딩 모델을 로드하는 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ 모델 로드 완료!


In [9]:
# 2. 추천 시스템의 영화 데이터베이스
movie_catalog = [
    {"id": 1, "title": "인터스텔라", "genre": "SF/우주", "desc": "우주 시공간을 탐험하며 인류를 구원하려는 우주비행사들의 이야기"},
    {"id": 2, "title": "라라랜드", "genre": "로맨스/뮤지컬", "desc": "꿈을 쫓는 재즈 피아니스트와 배우 지망생의 열정과 사랑 이야기"},
    {"id": 3, "title": "컨저링", "genre": "공포/스릴러", "desc": "한 가족이 이사 간 외딴 집에서 겪는 초자연적이고 무서운 현상"},
    {"id": 4, "title": "마션", "genre": "SF/우주", "desc": "화성에 홀로 고립된 우주비행사의 눈물겨운 생존기와 지구 귀환 프로젝트"},
    {"id": 5, "title": "노트북", "genre": "로맨스/드라마", "desc": "신분 차이를 극복하고 평생 동안 한 여자만을 사랑한 남자의 감동 실화"},
    {"id": 6, "title": "애나벨", "genre": "공포/스릴러", "desc": "저주받은 인형이 집에 들어온 후 벌어지는 기괴하고 끔찍한 사건들"}
]

# 3. 영화의 장르와 설명을 합쳐 임베딩 벡터 생성
catalog_texts = [f"장르: {m['genre']}. 내용: {m['desc']}" for m in movie_catalog]

# 오픈소스 모델은 복잡한 API 호출 없이 .encode() 함수 하나로 벡터를 추출합니다.
# ko-sroberta 모델은 768 차원의 벡터를 반환합니다.
print("🔄 영화 카탈로그 임베딩 벡터 변환 중...")
catalog_embeddings = model.encode(catalog_texts, convert_to_numpy=True)

🔄 영화 카탈로그 임베딩 벡터 변환 중...


In [10]:
# 4. FAISS 벡터 DB 구성 (768 차원에 맞춤 설정)
embedding_matrix = catalog_embeddings.astype('float32')
dimension = embedding_matrix.shape[1]  # 768

index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print(f"✅ 오픈소스 기반 벡터 DB 빌드 완료! (등록된 영화 개수: {index.ntotal}개)")
print("-" * 60)

✅ 오픈소스 기반 벡터 DB 빌드 완료! (등록된 영화 개수: 6개)
------------------------------------------------------------


In [11]:
# ----------------------------------------------------------------
# 5. [추천 시뮬레이션] 유저의 취향 입력
#user_history = "달콤하고 감동적인 로맨스 러브 스토리 영화가 보고 싶어"
#user_history = "우주비행사의 삶이 궁금해"
user_history = "무서운 영화"

print(f"🎯 유저 취향 입력: '{user_history}'")

# 유저 취향 문장도 동일한 오픈소스 모델로 임베딩 변환
user_taste_vector = model.encode([user_history], convert_to_numpy=True).astype('float32')

# 6. 벡터 DB에서 유저 취향과 가장 유사한 영화 TOP 2 검색
k = 2
distances, indices = index.search(user_taste_vector, k=k)

# 7. 추천 결과 출력
print(f"🎬 오픈소스 AI가 추천하는 TOP {k} 맞춤 영화:")

for i in range(k):
    match_idx = indices[0][i]
    distance_score = distances[0][i]
    recommended_movie = movie_catalog[match_idx]

    print(f"\n[{i+1}위 추천] {recommended_movie['title']} ({recommended_movie['genre']})")
    print(f"   ↳ 줄거리 요약: {recommended_movie['desc']}")
    print(f"   ↳ 취향 일치도 거릿값: {distance_score:.4f}")


🎯 유저 취향 입력: '무서운 영화'
🎬 오픈소스 AI가 추천하는 TOP 2 맞춤 영화:

[1위 추천] 애나벨 (공포/스릴러)
   ↳ 줄거리 요약: 저주받은 인형이 집에 들어온 후 벌어지는 기괴하고 끔찍한 사건들
   ↳ 취향 일치도 거릿값: 138.8584

[2위 추천] 컨저링 (공포/스릴러)
   ↳ 줄거리 요약: 한 가족이 이사 간 외딴 집에서 겪는 초자연적이고 무서운 현상
   ↳ 취향 일치도 거릿값: 165.4768


SentenceTransformers는 문장, 단락, 또는 이미지를 고차원의 수학적 좌표(밀집 벡터, Dense Vector Embedding)로 변환해 주는 가장 대표적인 파이썬 오픈소스 프레임워크입니다.
기존의 BERT 같은 일반적인 트랜스포머 모델은 단어 예측이나 문장 분류에는 뛰어났지만, "두 문장이 얼마나 비슷한지"를 계산할 때는 연산 속도가 치명적으로 느렸습니다. SentenceTransformers는 이 문제를 완벽하게 해결하기 위해 등장한 기술입니다 (학술적으로는 SBERT라고 불립니다).
이해하기 쉽게 핵심 원리와 특징을 3가지로 나누어 설명해 드릴게요.

------------------------------

## 1. 핵심 원리: 쌍둥이 신경망 (Siamese Network)
만약 1만 개의 문장 중에서 서로 가장 비슷한 문장 쌍을 찾으려면, 기존 BERT 모델은 모든 문장을 두 개씩 짝지어 모델에 입력해야 했습니다. 이 경우 약 5,000만 번의 AI 모델 연산이 필요해 몇 시간이 걸렸습니다.
SentenceTransformers는 쌍둥이 아키텍처를 사용합니다.

* 문장 A와 문장 B를 동일한 구조의 두 AI 모델에 각각 동시에 통과시킵니다.
* 모델을 거쳐 나온 출력값들을 평균(Pooling) 내어, 각 문장당 딱 하나의 고정된 크기의 벡터(숫자 배열)로 뽑아냅니다.
* 학습 단계에서 "의미가 비슷한 문장은 벡터 공간에서 거리가 가깝게, 의미가 다르면 멀어지게" 밀고 당기는 훈련(Contrastive Loss)을 거쳤습니다.

덕분에 우리는 1만 개의 문장을 처음에 딱 한 번만 벡터로 변환(.encode())해 두면, 이후에는 아주 단순한 고속 수학 연산(코사인 유사도 등)만으로 몇 밀리초(ms) 만에 의미 기반 검색을 끝낼 수 있습니다.
------------------------------
## 2. 왜 사용하나요? (장점)

* 🧠 진짜 의미를 이해하는 검색 (Semantic Search): 단어가 겹치지 않아도 맥락을 이해합니다. 예를 들어 *"오늘 날씨 정말 좋다"*라는 문장과 *"하늘이 맑고 화창하네"*라는 문장은 같은 단어가 하나도 없지만, 두 문장의 벡터 거리를 가깝게 매칭합니다.
* 🆓 100% 무료 및 로컬 실행: OpenAI 같은 API 서비스는 한 글자당 돈을 내야 하고 인터넷 연결이 필수지만, SentenceTransformers는 내 컴퓨터나 서버(Kaggle, Colab 등)의 GPU/CPU에서 완전 무료로 무제한 실행할 수 있습니다.
* 📦 수많은 사전 학습 모델 제공: 오픈소스 커뮤니티(Hugging Face)에 전 세계 연구자들이 미리 학습시켜 둔 수천 개의 모델이 있습니다. 특히 한국어 뉴스를 많이 학습한 모델, 법률 문서에 특화된 모델 등 목적에 맞게 골라 쓸 수 있습니다.
* ⚡ 압도적인 단순함: 개발자가 토큰화(Tokenization), 패딩(Padding), 패스 마스크 설계 같은 복잡한 딥러닝 전처리를 직접 코딩할 필요 없이, 단 한 줄의 .encode() 함수로 모든 과정을 끝내줍니다.

------------------------------
## 3. 주요 활용 요약

* RAG의 핵심 엔진: 문서를 벡터 DB에 채워 넣기 전, 문서를 숫자로 바꿀 때 무조건 사용됩니다.
* 추천 시스템: 상품 설명이나 유저 취향 문장을 벡터로 만들어 서로 매칭합니다.
* 중복 제거 및 분류: 대량의 고객 문의(CS) 데이터 중 똑같은 질문 내용이나 비슷한 카테고리의 글들을 묶어줄(Clustering) 때 쓰입니다.

------------------------------
## 💡 교육과정(커리큘럼)에서 설명하는 팁
이 개념은 파트 2(뉴럴 언어모델의 이해)의 마지막 부분이나 파트 5(RAG 시스템 만들기)의 시작 부분에 배치하면 완벽합니다.

"단어 레벨의 표현(Word2Vec)을 넘어, 문장 전체의 문맥적 의미를 하나의 벡터로 압축하는 기술이 왜 필요하며, 이를 통해 어떻게 LLM 없이도 초고속 유사도 검색을 할 수 있는지"를 설명하는 핵심 징검다리 역할을 합니다.

수강생들을 위한 강의 자료 생성을 돕기 위해, SentenceTransformers의 작동 원리를 직관적인 다이어그램(텍스트 형태)이나 슬라이드 요약본으로 정리해 드릴까요? 아니면 한국어 성능이 좋은 Hugging Face 모델 리스트를 추천해 드릴까요?

